# LSTMs with PyTorch — Sine Wave Prediction (SOLUTION)

LSTM adds a **cell state** `c_t` (long-term memory) and three gates to the vanilla RNN, solving the vanishing gradient problem.

Primary task: **time-series regression** on a synthetic sine wave.

## Step 1: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## Step 2: Generate Sine Wave

In [ ]:
t    = np.linspace(0, 4 * np.pi, 200)
data = np.sin(t).astype(np.float32)
print(f'data shape: {data.shape}  min={data.min():.2f}  max={data.max():.2f}')

## Step 3: Sliding Window Sequences

In [ ]:
def create_sequences(data, seq_len=20):
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        xs.append(data[i:i+seq_len].reshape(seq_len, 1))
        ys.append(data[i+seq_len].reshape(1))
    return torch.tensor(np.array(xs)), torch.tensor(np.array(ys))

X, y = create_sequences(data, seq_len=20)
print('X:', X.shape, '  y:', y.shape)

## Step 4: Train / Test Split

In [ ]:
split = int(0.8 * len(X))
X_train, X_test = X[:split].to(device), X[split:].to(device)
y_train, y_test = y[:split].to(device), y[split:].to(device)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## Step 5: Define the LSTM Model

`nn.LSTM` returns `(output, (h_n, c_n))` — the extra `c_n` (cell state) is the key difference from `nn.RNN`.

In [ ]:
class SineLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])   # last timestep

model = SineLSTM().to(device)
print(model)

## Step 6: Train

In [ ]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)
criterion    = nn.MSELoss()
optimizer    = optim.Adam(model.parameters(), lr=0.001)
epoch_losses = []

for epoch in range(100):
    model.train()
    total = 0
    for Xb, yb in train_loader:
        pred = model(Xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    avg = total / len(train_loader)
    epoch_losses.append(avg)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | Loss: {avg:.6f}')

## Step 7: Evaluate

In [ ]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test)
    test_loss = criterion(test_pred, y_test)
print(f'Test MSE: {test_loss.item():.6f}')

## Step 8: Visualize Predictions vs Actuals

In [ ]:
model.eval()
with torch.no_grad():
    preds   = model(X_test).cpu().squeeze().numpy()
    actuals = y_test.cpu().squeeze().numpy()

plt.figure(figsize=(12, 4))
plt.plot(actuals, label='Actual',    linewidth=2, color='#74b9ff')
plt.plot(preds,   label='Predicted', linewidth=2, linestyle='--', color='#fd79a8')
plt.legend()
plt.title('LSTM Sine Wave Prediction')
plt.xlabel('Test timestep')
plt.ylabel('Value')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 9: Compare LSTM vs vanilla RNN

In [ ]:
class SineRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 32, 2, batch_first=True)
        self.fc  = nn.Linear(32, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

rnn_model = SineRNN().to(device)
opt_rnn   = optim.Adam(rnn_model.parameters(), lr=0.001)
rnn_losses = []

for epoch in range(100):
    rnn_model.train()
    total = 0
    for Xb, yb in train_loader:
        pred = rnn_model(Xb)
        loss = criterion(pred, yb)
        opt_rnn.zero_grad()
        loss.backward()
        opt_rnn.step()
        total += loss.item()
    rnn_losses.append(total / len(train_loader))

rnn_model.eval()
with torch.no_grad():
    rnn_test_loss = criterion(rnn_model(X_test), y_test)

print(f'LSTM test MSE: {test_loss.item():.6f}')
print(f'RNN  test MSE: {rnn_test_loss.item():.6f}')

plt.figure(figsize=(8, 4))
plt.plot(epoch_losses, label='LSTM', color='#74b9ff', linewidth=2)
plt.plot(rnn_losses,   label='RNN',  color='#fd79a8', linewidth=2, linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('LSTM vs RNN — Training Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 10: Visualize the LSTM Cell Architecture

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('LSTM Cell: Gated Memory Mechanism', fontsize=14, fontweight='bold')

# Cell state highway (top)
ax.annotate('', xy=(13.5, 7.0), xytext=(0.5, 7.0),
            arrowprops=dict(arrowstyle='->', color='#e17055', lw=3))
ax.text(7, 7.4, 'Cell State  c_t  (“long-term memory”)', ha='center',
        fontsize=11, color='#e17055', fontweight='bold')

gates = [
    (2.5,  'Forget Gate\n(σ)', '#FAD7A0',
     'What to ERASE\nfrom memory\nf = σ(W[h,x]+b)'),
    (5.5,  'Input Gate\n(σ)', '#A9DFBF',
     'What NEW info\nto WRITE\ni = σ(W[h,x]+b)'),
    (8.5,  'Cell Update\n(tanh)', '#AED6F1',
     'Candidate values\nto add\ng = tanh(W[h,x]+b)'),
    (11.5, 'Output Gate\n(σ)', '#D7BDE2',
     'What to OUTPUT\nas h_t\no = σ(W[h,x]+b)'),
]
for x, label, color, desc in gates:
    rect = mpatches.FancyBboxPatch((x - 1.0, 3.6), 2.0, 1.8,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='#2d3436', lw=2)
    ax.add_patch(rect)
    ax.text(x, 4.6, label, ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(x, 3.2, desc, ha='center', va='top', fontsize=8, color='#636e72', linespacing=1.4)
    ax.annotate('', xy=(x, 6.8), xytext=(x, 5.4),
                arrowprops=dict(arrowstyle='->', color='#8e44ad', lw=1.5))

# h_{t-1} input
ax.annotate('', xy=(0.8, 4.5), xytext=(0.0, 4.5),
            arrowprops=dict(arrowstyle='->', color='#3498db', lw=2.5))
ax.text(-0.05, 4.5, 'h_{t-1}', ha='right', va='center', fontsize=10,
        color='#3498db', fontweight='bold')

# x_t inputs
ax.text(7, 2.1, 'x_t  (current input)', ha='center', fontsize=10,
        color='#27ae60', fontweight='bold')
for gx in [2.5, 5.5, 8.5, 11.5]:
    ax.annotate('', xy=(gx, 3.6), xytext=(gx, 2.7),
                arrowprops=dict(arrowstyle='->', color='#27ae60', lw=1.5))

# h_t output
ax.annotate('', xy=(14.0, 4.5), xytext=(13.0, 4.5),
            arrowprops=dict(arrowstyle='->', color='#3498db', lw=2.5))
ax.text(14.1, 4.5, 'h_t', ha='left', va='center', fontsize=10,
        color='#3498db', fontweight='bold')

plt.tight_layout()
plt.show()


## Bonus: Deep LSTM with Dropout

In [ ]:
class SineLSTMDeep(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, num_layers=3, batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(32, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

deep = SineLSTMDeep().to(device)
opt_deep = optim.Adam(deep.parameters(), lr=0.001)
deep_losses = []

for epoch in range(100):
    deep.train()
    total = 0
    for Xb, yb in train_loader:
        pred = deep(Xb)
        loss = criterion(pred, yb)
        opt_deep.zero_grad()
        loss.backward()
        opt_deep.step()
        total += loss.item()
    deep_losses.append(total / len(train_loader))

deep.eval()
with torch.no_grad():
    deep_loss = criterion(deep(X_test), y_test)
print(f'Deep LSTM (3-layer, dropout=0.2) test MSE: {deep_loss.item():.6f}')